In [1]:
import requests
from bs4 import BeautifulSoup, NavigableString
import pprint
import time
import json
import re

In [2]:
url_members = "https://philea.eu/wp-admin/admin-ajax.php"
payload = {"action":"phileamap"}
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"}

In [3]:
response = requests.post(url_members, data=payload, headers=headers)

In [4]:
members = response.json()["data"]
todo = len(members)
pprint.pprint(members)

[{'address': 'Rapenburgerstraat 173, Amsterdam, Netherlands',
  'country': '',
  'email': '',
  'id': 24788,
  'link': 'https://philea.eu/members/women-win/',
  'name': 'Women Win',
  'position': {'address': 'Rapenburgerstraat 173, Amsterdam, Netherlands',
               'city': 'Amsterdam',
               'country': 'Netherlands',
               'country_short': 'NL',
               'lat': 52.368106,
               'lng': 4.9061522,
               'name': 'Rapenburgerstraat 173',
               'place_id': 'ChIJe7GIF70JxkcRidx5mdy_5lo',
               'post_code': '1011 VM',
               'state': 'Noord-Holland',
               'state_short': 'NH',
               'street_name': 'Rapenburgerstraat',
               'street_number': '173',
               'zoom': 14},
  'type': {'label': 'Foundation and philanthropic organisation',
           'value': 'foundation'},
  'website': 'http://www.womenwin.org'},
 {'address': 'Untermüli 7, Zug, Switzerland',
  'country': '',
  'email': '',
  '

In [ ]:
counter = 0
for member in members:
    counter+=1
    print(f"Processing {counter}/{todo}: {member['name']}")
    philea_data = {}
    response = requests.get(member["link"], headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    content = soup.find("div",{"class":"article-body"})
    first_p = content.find("p")
    philea_data["About"] = first_p.text.strip() if first_p else ""
    h3s = content.find_all("h3")
    for h3 in h3s:
        key = h3.text.strip()
        value_parts = []
        
        # Wir wandern von der Überschrift aus vorwärts
        current = h3.next_sibling
        
        while current:
            # Wenn wir auf die NÄCHSTE Überschrift stoßen, stoppen wir sofort
            if current.name == "h3":
                break
                
            # Wenn es sich um reinen Text (ohne HTML-Tag) handelt
            if isinstance(current, NavigableString):
                text = current.strip()
                if text: # Verhindert das Aufnehmen von leeren Zeilenumbrüchen
                    value_parts.append(text)
                    
            # Wenn es ein HTML-Tag ist (p, ul, div, etc.), holen wir uns den Text
            elif current.name:
                text = current.text.strip()
                if text:
                    value_parts.append(text)
            
            # Gehe zum nächsten Geschwister-Element
            current = current.next_sibling
        
        # Füge alle gefundenen Textteile zusammen
        philea_data[key] = " ".join(value_parts)
    member["philea_info"] = philea_data
    time.sleep(1)

Processing 1/299: Women Win
Processing 2/299: Toni Piëch Foundation
Processing 3/299: Thomson Reuters Foundation


KeyboardInterrupt: 

In [42]:
pprint.pprint(members)

[{'address': 'Rapenburgerstraat 173, Amsterdam, Netherlands',
  'country': '',
  'email': '',
  'id': 24788,
  'link': 'https://philea.eu/members/women-win/',
  'name': 'Women Win',
  'position': {'address': 'Rapenburgerstraat 173, Amsterdam, Netherlands',
               'city': 'Amsterdam',
               'country': 'Netherlands',
               'country_short': 'NL',
               'lat': 52.368106,
               'lng': 4.9061522,
               'name': 'Rapenburgerstraat 173',
               'place_id': 'ChIJe7GIF70JxkcRidx5mdy_5lo',
               'post_code': '1011 VM',
               'state': 'Noord-Holland',
               'state_short': 'NH',
               'street_name': 'Rapenburgerstraat',
               'street_number': '173',
               'zoom': 14},
  'type': {'label': 'Foundation and philanthropic organisation',
           'value': 'foundation'},
  'website': 'http://www.womenwin.org'},
 {'address': 'Untermüli 7, Zug, Switzerland',
  'country': '',
  'email': '',
  '

In [53]:
# Wirte to file
import json
with open("philea_members.json", "w", encoding="utf-8") as f:
    json.dump(members, f, ensure_ascii=False, indent=4)

### Preprocessing

In [61]:
# Load data
with open("philea_members.json", "r", encoding="utf-8") as f:
    members = json.load(f)
print(f"Loaded {len(members)} members")

Loaded 299 members


In [62]:
# Extract all unique keys from philea_info
unique_keys = set()
for member in members:
    unique_keys.update(member.get("philea_info", {}).keys())
print(f"Unique keys in philea_info: {unique_keys}")

Unique keys in philea_info: {'About', 'Mission', 'Programme Areas', 'Geographic Focus'}


In [71]:
# 1. Alle Roh-Tags, die jemals in den Philea-Daten auftauchen können (inkl. Human/Civil Rights)
MASTER_TAGS = [
    "Citizenship, Social Justice & Public Affairs",
    "Civil society, Voluntarism & Non-Profit Sector",
    "Socio-economic Development, Poverty",
    "Socio-economic Development",
    "Food, Agriculture & Nutrition",
    "Recreation, Sport & Well-being",
    "Humanitarian & Disaster Relief",
    "Peace & Conflict Resolution",
    "Youth/Children Development",
    "Sciences & Research",
    "Employment/Workforce",
    "Environment/Climate",
    "Social/Human Services",
    "Arts & Culture",
    "Arts and Culture",
    "Policy development",
    "Education",
    "Health",
    "Animal-Related",
    "Water",
    "Nature",
    "Human/Civil Rights"  # FIX: War vorher vergessen
]

# 2. Die zentrale Mapping-Schmiede (Konsolidiert von ~24 auf 13 Hauptkategorien)
TAG_NORMALIZATION = {
    # Schreibweisen-Korrekturen
    "Arts and Culture": "Arts & Culture",
    "Socio-economic Development": "Socio-economic Development, Poverty",
    "Environment": "Environment/Climate",
    
    # Strategische Zusammenfassungen für ein sauberes Datenmodell
    "Nature": "Environment/Climate",
    "Water": "Environment/Climate",
    "Animal-Related": "Environment/Climate",
    "Employment/Workforce": "Socio-economic Development, Poverty",
    "Social/Human Services": "Socio-economic Development, Poverty",
    "Recreation, Sport & Well-being": "Health",
    "Policy development": "Citizenship, Social Justice & Public Affairs"
}

# 3. Die Keyword-Kanten für den Freitext-Fallback (Exakt synchron zu den Normalisierungs-Targets)
KEYWORD_MAPPING = {
    "Environment/Climate": [
        r"climates?", r"emissions?", r"carbon", r"energy transition", r"fossil fuels?", 
        r"biodiversity", r"nature conservation", r"planet", r"agroecology", r"built environment",
        r"plastic( pollution)?", r"petrochemical", r"economies of reuse", r"ocean economy", r"maritime", r"gardens?",
        r"animals?", r"wildlife", r"water security", r"water supply" # FIX: Fängt 'Animal-Related', 'Nature' & 'Water' ab
    ],
    "Education": [
        r"educations?", r"learnings?", r"schools?", r"scholarships?", r"students?", r"stem", r"teachers?", r"trainings?"
    ],
    "Arts & Culture": [
        r"arts?", r"culture?", r"cultural", r"museums?", r"exhibitions?", r"music", r"artists?", r"heritage", r"theatres?", r"villa"
    ],
    "Citizenship, Social Justice & Public Affairs": [
        r"democrac\w+", r"civil societ\w+", r"civic", r"citizenship", r"public affairs", 
        r"press freedom", r"independent media", r"advocacy", r"journalism", r"newsrooms?",
        r"social cohesion", r"responsible leadership", r"criminal justice", r"social change", r"polycrisis",
        r"policy development" # FIX: Fängt 'Policy development' ab
    ],
    "Human/Civil Rights": [
        r"human rights", r"civil rights", r"gender equality", r"women’s rights", 
        r"lgbti\+", r"feminist", r"discrimination", r"gender justice"
    ],
    "Youth/Children Development": [
        r"children", r"youths?", r"child", r"young people", r"early childhood", r"infants?", r"neonatal", r"0-5 year olds"
    ],
    "Socio-economic Development, Poverty": [
        r"poverty", r"low-income", r"vulnerabilit\w+", r"marginalized", r"homeless\w*", 
        r"social inclusion", r"disadvantaged", r"social justice", r"social innovation",
        r"economic justice", r"social leaders?", r"social development", r"economic development",
        r"employ\w+", r"workforce", r"jobs?", r"labour", r"social services?" # FIX: Fängt 'Employment' & 'Social Services' ab
    ],
    "Health": [
        r"health\w*", r"medical", r"diseases?", r"healthcare", r"illness\w*", r"pain therapy", r"sanitation",
        r"sports?", r"recreation", r"well[- ]being" # FIX: Fängt 'Recreation, Sport & Well-being' ab
    ],
    "Sciences & Research": [
        r"research\w*", r"scientific", r"sciences?", r"phd", r"academia", r"universit\w+"
    ],
    "Food, Agriculture & Nutrition": [
        r"food", r"agriculture", r"nutrition", r"farming", r"diets?"
    ],
    "Humanitarian & Disaster Relief": [
        r"disaster relief", r"humanitarian", r"emergency response", r"refugees?", r"asylum seekers?", r"migration", r"foreign aid"
    ],
    "Civil society, Voluntarism & Non-Profit Sector": [
        r"philanthrop\w+", r"fundraising", r"donors?", r"grant-making", r"fiscal sponsorship"
    ],
    "Peace & Conflict Resolution": [
        r"peacebuilding", r"conflict sensitivity", r"peace work"
    ]
}

def extract_tags_final(raw_text):
    if not raw_text:
        return []
        
    found_tags = set()
    text_lower = raw_text.lower()
    
    for tag, keywords in KEYWORD_MAPPING.items():
        for kw in keywords:
            # Nutzt nun die kompilierten Regex-Patterns mit Wortgrenzen \b
            pattern = r'\b' + kw + r'\b'
            if re.search(pattern, text_lower):
                found_tags.add(tag)
                # Wir brechen hier nicht ab, damit eine Stiftung mehrere Tags erhalten kann
                
    return sorted(list(found_tags))

def extract_tags_robust(raw_text):
    if not raw_text:
        return []
        
    # Tags stehen bei Philea IMMER ganz am Anfang vor dem Fließtext.
    # Wir untersuchen daher nur die ersten 600 Zeichen (Sicherheitsfenster).
    zone = raw_text[:600].lower()
    
    # Text-Normalisierung: Zeilenumbrüche und alle Arten von Bindestrichen/Bulletpoints 
    # durch einfache Leerzeichen ersetzen. Kommas bleiben als Trenner erhalten.
    zone_clean = re.sub(r'[\s\–\—\-]+', ' ', zone)
    
    found_tags = set()
    
    # Wichtig: Wir sortieren nach Länge (absteigend), damit lange Phrasen wie
    # "Socio-economic Development, Poverty" vor "Education" oder "Health" gematched werden.
    sorted_master_tags = sorted(MASTER_TAGS, key=len, reverse=True)
    
    for original_tag in sorted_master_tags:
        tag_clean = original_tag.lower()
        tag_clean = re.sub(r'[\s\–\—\-]+', ' ', tag_clean)
        
        if tag_clean in zone_clean:
            # Sicherheits-Check für sehr kurze Tags (z.B. "Health"), damit sie nicht 
            # fälschlicherweise in Wörtern wie "Healthcare" oder "Healthy" matchen.
            if len(tag_clean) <= 10:
                # Regex prüft, ob vor und nach dem Tag kein Buchstabe (a-z) steht
                pattern = r'(?<![a-z])' + re.escape(tag_clean) + r'(?![a-z])'
                if re.search(pattern, zone_clean):
                    found_tags.add(original_tag)
                    # Löschen, um Doppel-Treffer zu vermeiden
                    zone_clean = zone_clean.replace(tag_clean, " ")
            else:
                found_tags.add(original_tag)
                zone_clean = zone_clean.replace(tag_clean, " ")
                
    # Normalisieren (z.B. "Arts and Culture" -> "Arts & Culture")
    final_tags = [TAG_NORMALIZATION.get(t, t) for t in found_tags]
    
    return sorted(list(set(final_tags)))

# Loop über deine Struktur
parsed_results = {}
for member in members:
    tags = extract_tags_robust(member.get("philea_info", {}).get("Programme Areas", ""))
    if not tags: # Fallback auf die Freitext-Analyse, wenn keine Tags gefunden wurden
        tags = extract_tags_final(member.get("philea_info", {}).get("Programme Areas", ""))
    if not tags: # Letzte Chance: Manchmal stehen die Tags nicht unter "Programme Areas", sondern nur im allgemeinen "About"-Text
        tags = extract_tags_final(member.get("philea_info", {}).get("About", ""))
    if not tags:
        tags = extract_tags_final(member.get("philea_info", {}).get("Mission", ""))
    parsed_results[member["name"]] = tags

# Ergebnis anzeigen
#pprint.pprint(parsed_results)
print(len(parsed_results))
# Anzahl leerer Einträge
empty_count = sum(1 for tags in parsed_results.values() if not tags)
print(f"Anzahl Mitglieder ohne Tags: {empty_count}")
for member in members:
    member["tags_focus"] = sorted(parsed_results.get(member["name"], []))

299
Anzahl Mitglieder ohne Tags: 20


In [72]:
pprint.pprint(members)

[{'address': 'Rapenburgerstraat 173, Amsterdam, Netherlands',
  'country': '',
  'email': '',
  'geo_locations': {'Global South / Majority World': ['Global South'],
                    'Worldwide': ['Global']},
  'id': 24788,
  'link': 'https://philea.eu/members/women-win/',
  'name': 'Women Win',
  'philea_info': {'About': 'Women Win operates as a multidimensional global '
                           'women’s fund with expertise across three '
                           'interconnected areas:',
                  'Geographic Focus': 'Women Win is operates globally, but '
                                      'with a majority focus on Global South '
                                      'and East.',
                  'Mission': 'Women Win is a global multidimensional women’s '
                             'fund guided by the vision of a future where '
                             'every girl and woman exercises their rights. We '
                             'work towards our vision thro

In [65]:
# Gleiches jetzt auch für Geographic:
for member in members:
    print("---", member["name"], "---")
    print(member.get("philea_info", {}).get("Geographic Focus", ""))

--- Women Win ---
Women Win is operates globally, but with a majority focus on Global South and East.
--- Toni Piëch Foundation ---
Global
--- Thomson Reuters Foundation ---
Global
--- The Social Change Nest ---
We tear down the barriers that prevent communities from creating change. We take care of the finance and administration, freeing groups to focus on their core mission, and work closely with funders to enable them to support social impact with confidence and transparency.
--- The Global Fund for Children ---
Global
--- Suna and İnan Kıraç Foundation ---
Currently, we are exclusively in Turkiye, though our long-term vision is to extend to neighboring countries, particularly with our education work.
--- Hidden Universe: Biodiversity Foundation (HUB) ---
Brazil (initial focus).
--- Andrea von Braun Stiftung ---
Europe and Germany, but also worldwide
--- Philanthropy in Ukraine ---
Currently, we operate in Ukraine for Ukrainian organizations, but we also provide services and have pa

In [ ]:
import re
import pprint

# =====================================================================
# SINGLE SOURCE OF TRUTH: Makro-Region -> { Anzeige-Name: Such-Regex }
# =====================================================================
GEO_TAXONOMY = {
    "Worldwide": {
        "Global": r"global\w*",
        "Worldwide": r"worldwide",
        "World": r"\bworld\b"
    },
    "Global South / Majority World": {
        "Global South": r"global south",
        "Majority World": r"majority world",
        "Developing Countries": r"developing world|developing countr\w+|low and middle income"
    },
    "Europe (Western / General)": {
        "Europe": r"europ\w+",
        "European Union": r"european union|\beu\b",
        "United Kingdom": r"\buk\b|united kingdom|great britain|london|scotland|west midlands|english", 
        "Ireland": r"ireland|irish",
        "France": r"franc\w+",
        "Germany": r"german\w+",
        "Switzerland": r"switzerland|swiss",
        "Austria": r"austria\w*",
        "Luxembourg": r"luxembourg\w*",
        "Belgium": r"belgium\w*|belgian\w*|brussels",
        "Netherlands": r"netherlands|dutch|the hague|delft|zoetermeer|leiden|noordwijk"
    },
    "Europe (Nordic Region)": {
        "Nordic Region": r"nordic",
        "Denmark": r"denmark|danish",
        "Finland": r"finland|finnish|herlin", # fängt Herlin-Stiftung ab
        "Sweden": r"sweden|swedish|\bse\b",
        "Norway": r"norway|norwegian|kristiansand",
        "Greenland": r"greenland",
        "Faroe Islands": r"faroe islands"
    },
    "Europe (Southern / Mediterranean)": {
        "Spain": r"spain|spanish|galicia",
        "Italy": r"ital\w+|sicily|sardinia|piedmont|aosta valley|modena|parma|padua|rovigo|tuscany|florence|grosseto|arezzo|cuneo|alto adige|lucca|lombardy|torino|bologna",
        "Greece": r"gree\w+",
        "Portugal": r"portug\w+",
        "Turkey": r"turk\w+|türkiye"
    },
    "Europe (Central & Eastern / Balkans)": {
        "Balkans": r"balkans?|western balkans|serbian?|croatian?|slovenian?|bosnia\w*",
        "Central & Eastern Europe": r"cee\b|eastern europe|central and eastern europe|baltic\w*",
        "Slovakia": r"slovakia\w*",
        "Bulgaria": r"bulgari\w*",
        "Kosovo": r"kosovo\w*",
        "Croatia": r"croatia\w*",
        "Slovenia": r"slovenia\w*",
        "Ukraine": r"ukrain\w*",
        "Estonia": r"estonia\w*",
        "Lithuania": r"lithuania\w*",
        "Poland": r"pol\w+|fundacja", # 'fundacja' deutet direkt auf Polen hin
        "Latvia": r"latvia\w*",
        "Georgia": r"georgia\w*",
        "Czech Republic": r"czech\w*",
        "Romania": r"romani\w*",
        "Hungary": r"hungar\w*",
        "Belarus": r"belarus\w*",
        "Moldova": r"moldova\w*"
    },
    "North America": {
        "United States": r"united states|\busa\b|\bus\b|america\w*|flint|michigan",
        "Canada": r"canada\w*"
    },
    "Latin America & Caribbean": {
        "Latin America": r"latin america|south america|central america",
        "Caribbean": r"caribbean",
        "Brazil": r"brazil\w*",
        "Mexico": r"mexico\w*",
        "Colombia": r"colombia\w*",
        "Peru": r"peru\w*",
        "Bolivia": r"bolivia\w*",
        "Ecuador": r"ecuador\w*",
        "Guyana": r"guyana\w*"
    },
    "Africa / Sub-Saharan Africa": {
        "Africa": r"afric\w+",
        "Sub-Saharan Africa": r"sub-saharan",
        "Tanzania": r"tanzania\w*",
        "Kenya": r"kenya\w*",
        "Ethiopia": r"ethiopia\w*",
        "Uganda": r"uganda\w*",
        "Malawi": r"malawi\w*",
        "Ghana": r"ghana\w*",
        "Burkina Faso": r"burkina faso",
        "Zambia": r"zambia\w*",
        "Sierra Leone": r"sierra leone",
        "Madagascar": r"madagascar\w*",
        "Rwanda": r"rwanda\w*",
        "Zimbabwe": r"zimbabwe\w*",
        "South Africa": r"south africa|botswana|namibia|senegal|gambia|togo|benin|mali"
    },
    "Asia & Pacific": {
        "Asia": r"asia\w*",
        "Pacific": r"pacific",
        "India": r"india\w*",
        "China": r"china\w*",
        "Vietnam": r"vietnam\w*",
        "Cambodia": r"cambodia\w*",
        "Laos": r"laos\w*",
        "Myanmar": r"myanmar\w*",
        "Thailand": r"thailand\w*",
        "Nepal": r"nepal\w*",
        "Sri Lanka": r"sri lanka",
        "Indonesia": r"indonesia\w*",
        "Bangladesh": r"bangladesh\w*",
        "Philippines": r"philippines?",
        "Afghanistan": r"afghanistan\w*",
        "Australia": r"australia\w*|singapore"
    },
    "Middle East & North Africa (MENA)": {
        "Middle East": r"middle east",
        "MENA": r"mena\b",
        "Arab World": r"arab world",
        "Israel": r"israel\w*",
        "Palestine": r"palestin\w+",
        "Yemen": r"yemen\w*"
    }
}

# HILFS-STRUKTUREN AUTOMATISCH GENERIERT (Keine doppelte Pflege nötig!)
ALL_COUNTRIES = []
COUNTRY_TO_MACRO = {}
for macro_region, country_dict in GEO_TAXONOMY.items():
    for country_name in country_dict.keys():
        ALL_COUNTRIES.append(country_name)
        COUNTRY_TO_MACRO[country_name] = macro_region

# Sortierung nach Länge (absteigend) schützt "Sub-Saharan Africa" vor "Africa"
SORTED_COUNTRIES = sorted(ALL_COUNTRIES, key=len, reverse=True)


def extract_geos_robust(raw_text):
    """ Stufe 1: Sucht nach exakten Begriffen im Header-Bereich """
    if not raw_text:
        return {}
        
    zone = raw_text.lower()
    zone_clean = re.sub(r'[\s\–\—\-]+', ' ', zone)
    found = {}
    
    for country in SORTED_COUNTRIES:
        c_clean = country.lower()
        if c_clean in zone_clean:
            is_match = False
            if len(c_clean) <= 5: # Für kurze Token wie UK, US, Spain
                pattern = r'(?<![a-z])' + re.escape(c_clean) + r'(?![a-z])'
                if re.search(pattern, zone_clean):
                    is_match = True
            else:
                is_match = True
                
            if is_match:
                macro = COUNTRY_TO_MACRO[country]
                if macro not in found:
                    found[macro] = set()
                found[macro].add(country)
                zone_clean = zone_clean.replace(c_clean, " ") # Konsumieren
                
    return {k: sorted(list(v)) for k, v in found.items()}


def extract_geos_final(raw_text):
    """ Stufe 2: Tiefe Regex-Suche im gesamten Freitext """
    if not raw_text:
        return {}
        
    found = {}
    text_lower = raw_text.lower()
    
    for macro, country_dict in GEO_TAXONOMY.items():
        for country_name, pattern in country_dict.items():
            regex_pattern = r'\b' + pattern + r'\b'
            if re.search(regex_pattern, text_lower):
                if macro not in found:
                    found[macro] = set()
                found[macro].add(country_name)
                
    return {k: sorted(list(v)) for k, v in found.items()}

# =====================================================================
# PIPELINE EXECUTION
# =====================================================================

parsed_geo_results = {}

for member in members:
    raw_geo_text = member.get("philea_info", {}).get("Geographic Focus", "")
    
    # Kaskade abfeuern
    geos = extract_geos_robust(raw_geo_text)
    if not geos:
        geos = extract_geos_final(raw_geo_text)
        
    parsed_geo_results[member["name"]] = geos

# Zurückschreiben in dein Haupt-Objekt
for member in members:
    member["geo_locations"] = parsed_geo_results.get(member["name"], {})

In [67]:
pprint.pprint(members)

[{'address': 'Rapenburgerstraat 173, Amsterdam, Netherlands',
  'country': '',
  'email': '',
  'geo_locations': {'Global South / Majority World': ['Global South'],
                    'Worldwide': ['Global']},
  'id': 24788,
  'link': 'https://philea.eu/members/women-win/',
  'name': 'Women Win',
  'philea_info': {'About': 'Women Win operates as a multidimensional global '
                           'women’s fund with expertise across three '
                           'interconnected areas:',
                  'Geographic Focus': 'Women Win is operates globally, but '
                                      'with a majority focus on Global South '
                                      'and East.',
                  'Mission': 'Women Win is a global multidimensional women’s '
                             'fund guided by the vision of a future where '
                             'every girl and woman exercises their rights. We '
                             'work towards our vision thro

In [68]:
# Write final output to file
with open("philea_members_preprocessed.json", "w", encoding="utf-8") as f:
    json.dump(members, f, ensure_ascii=False, indent=4)
        